In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn

# GRU model architecture (must match training).
class BasicGRURUL(nn.Module):
    def __init__(self, input_size, gru1_hidden=100, gru2_hidden=50, dropout=0.2):
        super().__init__()
        self.gru1 = nn.GRU(input_size=input_size, hidden_size=gru1_hidden, batch_first=True)
        self.dropout1 = nn.Dropout(dropout)

        self.gru2 = nn.GRU(input_size=gru1_hidden, hidden_size=gru2_hidden, batch_first=True)
        self.dropout2 = nn.Dropout(dropout)

        self.fc = nn.Linear(gru2_hidden, 1)

    def forward(self, x):
        out, _ = self.gru1(x)
        out = self.dropout1(out)

        out, _ = self.gru2(out)
        out = self.dropout2(out)

        last = out[:, -1, :]
        return self.fc(last)

def gen_sequence(id_df, seq_length, seq_cols):
    data_array = id_df[seq_cols].values
    n = data_array.shape[0]
    for start, stop in zip(range(0, n - seq_length), range(seq_length, n)):
        yield data_array[start:stop, :]

def build_test_sequence_data(test_df, seq_length, label_col="RUL", seq_cols=None):
    if seq_cols is None:
        seq_cols = [c for c in test_df.columns if c not in ["id", "cycle", label_col]]

    X_test = np.array([
        test_df[test_df["id"] == eid][seq_cols].values[-seq_length:]
        for eid in test_df["id"].unique()
        if len(test_df[test_df["id"] == eid]) >= seq_length
    ]).astype(np.float32)

    mask = [len(test_df[test_df["id"] == eid]) >= seq_length for eid in test_df["id"].unique()]
    y_test = test_df.groupby("id")[label_col].nth(-1)[mask].values.astype(np.float32).reshape(-1, 1)

    return seq_cols, X_test, y_test

In [3]:
# Load exactly one checkpoint.
checkpoint_path = Path("./saved_models/gru_seed_999/gru_fe2_lv1_seq30_lr1e-03_seed999.pt")

if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

print(f"Using checkpoint: {checkpoint_path}")

# Load test dataset (FD001 low variance set used in training).
test_df = pd.read_csv(
    "../../data/processed-nasa-data/feature_engineering_2/low_variance_1/test_fd001_low_variance_1_125.csv"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location=device)
print({k: checkpoint[k] for k in ["dataset_name", "dataset_label", "seq_len", "learning_rate", "seed"] if k in checkpoint})

Using checkpoint: saved_models\gru_seed_999\gru_fe2_lv1_seq30_lr1e-03_seed999.pt
{'dataset_name': 'fe2_lv1', 'dataset_label': 'FD001_low_variance_1', 'seq_len': 30, 'learning_rate': 0.001, 'seed': 999}


In [4]:
# Rebuild test windows using checkpoint metadata.
seq_len = int(checkpoint["seq_len"])
feature_columns = checkpoint.get("feature_columns")

seq_cols, X_test, y_test = build_test_sequence_data(
    test_df=test_df,
    seq_length=seq_len,
    label_col="RUL",
    seq_cols=feature_columns,
)

print("X_test:", X_test.shape, "y_test:", y_test.shape)

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_t = torch.tensor(y_test, dtype=torch.float32).to(device)

# Recreate model from checkpoint hyperparameters and load weights.
model = BasicGRURUL(
    input_size=int(checkpoint["input_size"]),
    gru1_hidden=int(checkpoint["gru1_hidden"]),
    gru2_hidden=int(checkpoint["gru2_hidden"]),
    dropout=float(checkpoint["dropout"]),
).to(device)
model.load_state_dict(checkpoint["state_dict"])
model.eval()

X_test: (100, 30, 17) y_test: (100, 1)


BasicGRURUL(
  (gru1): GRU(17, 100, batch_first=True)
  (dropout1): Dropout(p=0.2, inplace=False)
  (gru2): GRU(100, 50, batch_first=True)
  (dropout2): Dropout(p=0.2, inplace=False)
  (fc): Linear(in_features=50, out_features=1, bias=True)
)

In [ ]:
# Evaluate on test set.
criterion = nn.MSELoss()
with torch.no_grad():
    y_pred_t = model(X_test_t)
    test_mse = criterion(y_pred_t, y_test_t).item()
    test_rmse = float(np.sqrt(test_mse))

print(f"Test RMSE: {test_rmse:.4f}")

# Example to see first 10 preds
pred_df = pd.DataFrame(
    {
        "y_true": y_test_t.cpu().numpy().ravel(),
        "y_pred": y_pred_t.cpu().numpy().ravel(),
    }
)
display(pred_df.head(10))

Test RMSE: 13.2656


,y_true,y_pred
0,112.0,113.577705
1,98.0,119.852051
2,69.0,50.235897
3,82.0,79.913322
4,91.0,104.609352
5,93.0,111.960091
6,91.0,109.214760
7,95.0,91.394600
8,111.0,117.591064
9,96.0,88.595116
